In [1]:
# Import Models

import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [3]:
df_X6 = pd.read_csv("homework_6.1.csv")

In [6]:
treated = df_X6[df_X6["X"] == 1].reset_index(drop=True)
control = df_X6[df_X6["X"] == 0].reset_index(drop=True)

Z_treated = treated[["Z"]].values
Z_control = control[["Z"]].values

# ---------------------------------------------------------
# Nearest-neighbor matching on the confounder Z
# ---------------------------------------------------------
# Each treated unit -> nearest control unit
nn_control = NearestNeighbors(n_neighbors=1).fit(Z_control)
_, idx_t2c = nn_control.kneighbors(Z_treated)

# Each control unit -> nearest treated unit
nn_treated = NearestNeighbors(n_neighbors=1).fit(Z_treated)
_, idx_c2t = nn_treated.kneighbors(Z_control)

treated_effects = treated["Y"].values - control["Y"].values[idx_t2c.flatten()]
control_effects = treated["Y"].values[idx_c2t.flatten()] - control["Y"].values

# ---------------------------------------------------------
# Q1: Average Treatment Effect (ATE)
# Every unit (treated + control) paired with its nearest
# opposite-group counterfactual; average all differences.
# ---------------------------------------------------------
ATE = np.concatenate([treated_effects, control_effects]).mean()
print(f"Q1 - ATE:  {ATE:.4f}")   # -> 1.695  (Option B)

# ---------------------------------------------------------
# Q2: Average Treatment Effect on the Treated (ATT)
# Only treated units, each matched to nearest control.
# ---------------------------------------------------------
ATT = treated_effects.mean()
print(f"Q2 - ATT:  {ATT:.4f}")   # -> 1.846  (Option B)

# ---------------------------------------------------------
# Q3: Average Treatment Effect on the Untreated (ATU)
# Only control units, each matched to nearest treated.
# ---------------------------------------------------------
ATU = control_effects.mean()
print(f"Q3 - ATU:  {ATU:.4f}")   # -> 1.549  (Option B)

# ---------------------------------------------------------
# Q4: Optimal treatment effect
# The single largest effect among untreated->treated matched pairs.
# ---------------------------------------------------------
optimal = control_effects.max()
print(f"Q4 - Optimal: {optimal:.4f}")  # -> 2.172  (Option C)

Q1 - ATE:  1.6953
Q2 - ATT:  1.8464
Q3 - ATU:  1.5495
Q4 - Optimal: 2.1725
